# AgriVision : DINOv3 ViT-S/16 gelé + tête de segmentation sur Agriculture-Vision 2021

**Prérequis (une seule fois)**
1. Demander l'accès au modèle sur https://huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m (modèle *gated*, accepter la licence DINOv3).
2. Créer un token HF (lecture) sur https://huggingface.co/settings/tokens, puis l'ajouter dans les **Secrets Colab** (icône 🔑) sous le nom `HF_TOKEN`, avec l'accès notebook activé.
3. Runtime → Modifier le type d'exécution → GPU **L4** ou **A100**.

La première exécution télécharge l'archive de 21 Go depuis S3 (bucket public AWS Open Data) et construit un sous-ensemble équilibré, qui est sauvegardé sur Drive. Les sessions suivantes repartent de ce sous-ensemble en quelques minutes.

In [ ]:
# --- Configuration ---
DRIVE_DIR = "/content/drive/MyDrive/agrivision"     # sous-ensemble + runs (persistant)
CODE_SOURCE = "drive"                                # "drive" : copie depuis CODE_DIR ; "git" : clone REPO_URL
CODE_DIR = f"{DRIVE_DIR}/code"                       # dossier du projet (contient agrivision/)
REPO_URL = "https://github.com/<toi>/AgriVision.git"

N_TRAIN, N_VAL, N_TEST = 12000, 3000, 2000
RUN_NAME = "vits16_frozen_512"
EPOCHS, BATCH_SIZE, LR = 20, 32, 1e-3
BACKBONE = "facebook/dinov3-vits16-pretrain-lvd1689m"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess
os.makedirs(DRIVE_DIR, exist_ok=True)
if CODE_SOURCE == "git":
    !rm -rf /content/AgriVision && git clone -q {REPO_URL} /content/AgriVision
else:
    !rm -rf /content/AgriVision && cp -r "{CODE_DIR}" /content/AgriVision
%cd /content/AgriVision
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# Connexion HF + vérification de l'accès au modèle gated
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))
from transformers import AutoConfig
print(AutoConfig.from_pretrained(BACKBONE))

## Données
Si `agv_subset.tar` existe sur Drive, on le recopie simplement. Sinon, on télécharge depuis S3 en extrayant à la volée (sans le test officiel et sans le NIR), puis on construit le sous-ensemble. Compter environ 30 à 60 min la première fois.

In [ ]:
SUBSET_TAR = f"{DRIVE_DIR}/agv_subset.tar"
DATA = "/content/agv_subset"
S3 = "https://intelinair-data-releases.s3.amazonaws.com/agriculture-vision/cvpr_challenge_2021/supervised/Agriculture-Vision-2021.tar.gz"

if os.path.exists(SUBSET_TAR):
    !tar xf "{SUBSET_TAR}" -C /content
else:
    !mkdir -p /content/raw
    !curl -sL {S3} | tar xzf - -C /content/raw --exclude='*/test/*' --exclude='*/images/nir/*'
    !du -sh /content/raw/Agriculture-Vision-2021/*
    !python -m agrivision.prepare_subset --src /content/raw/Agriculture-Vision-2021 --out {DATA} \
        --n-train {N_TRAIN} --n-val {N_VAL} --n-test {N_TEST} --workers {os.cpu_count()}
    !tar cf "{SUBSET_TAR}" -C /content agv_subset
    !rm -rf /content/raw
!du -sh {DATA}/* && cat {DATA}/meta.json | head -40

## Entraînement
Les checkpoints (`best.pt`, `last.pt`) et `log.jsonl` sont écrits sur Drive. Si la session coupe, il suffit de relancer la cellule : l'entraînement reprend depuis `last.pt`.

In [ ]:
RUN = f"{DRIVE_DIR}/runs/{RUN_NAME}"
!python -m agrivision.train --data {DATA} --out "{RUN}" --backbone {BACKBONE} \
    --epochs {EPOCHS} --batch-size {BATCH_SIZE} --lr {LR} --workers {os.cpu_count()}

In [ ]:
# Évaluation finale sur le test (parcelles du val officiel jamais vues pendant la sélection)
!python -m agrivision.train --data {DATA} --out "{RUN}" --backbone {BACKBONE} --eval-only --split test --workers {os.cpu_count()}

In [ ]:
import json, matplotlib.pyplot as plt
log = [json.loads(l) for l in open(f"{RUN}/log.jsonl")]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot([e["loss"] for e in log]); ax[0].set_title("loss train"); ax[0].set_xlabel("epoch")
for c in log[0]["val_iou"]:
    ax[1].plot([e["val_iou"][c] for e in log], label=c)
ax[1].plot([e["val_miou"] for e in log], "k--", lw=2, label="mIoU")
ax[1].set_title("IoU val"); ax[1].legend(fontsize=7, ncol=2); plt.show()

In [ ]:
# Visualisation : image | vérité terrain | prédiction
import numpy as np, torch
from agrivision import CLASSES, MEAN, STD
from agrivision.data import AgriVisionSubset, decode_labels
from agrivision.model import DinoSegmenter

device = "cuda"
model = DinoSegmenter(len(CLASSES), backbone=BACKBONE).to(device).eval()
model.load_state_dict(torch.load(f"{RUN}/best.pt", map_location=device)["model"], strict=False)
ds = AgriVisionSubset(DATA, "test")

PALETTE = np.array([[0,0,0],[230,25,75],[255,225,25],[60,180,75],[245,130,48],
                    [145,30,180],[0,130,200],[70,240,240],[240,50,230]], np.uint8)

def show(idxs):
    fig, axes = plt.subplots(len(idxs), 3, figsize=(12, 4 * len(idxs)))
    for row, i in zip(axes, idxs):
        img, lbl = ds[i]
        target, valid = decode_labels(lbl[None].to(device), len(CLASSES))
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
            pred = model(img[None].to(device)).argmax(1)[0].cpu().numpy()
        gt = (target[0].float() * torch.arange(len(CLASSES), device=device)[:, None, None]).amax(0).cpu().numpy().astype(int)
        rgb = (img.permute(1, 2, 0).numpy() * STD + MEAN).clip(0, 1)
        inv = ~valid[0].cpu().numpy()
        gt_rgb, pred_rgb = PALETTE[gt], PALETTE[pred]
        gt_rgb[inv] = pred_rgb[inv] = 128
        for a, im, t in zip(row, [rgb, gt_rgb, pred_rgb], [ds.ids[i], "vérité", "prédiction"]):
            a.imshow(im); a.set_title(t, fontsize=8); a.axis("off")
    plt.tight_layout(); plt.show()
    print("  ".join(f"{c}={tuple(PALETTE[k])}" for k, c in enumerate(CLASSES)))

show(np.random.choice(len(ds), 4, replace=False))